In [1]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
import feature_eng as fe
CONFIG_PATH = "../config.yaml"

In [2]:
gold = pl.read_parquet("../data/processed/gold_cleaned.parquet")
nifty = pl.read_parquet("../data/processed/nifty_cleaned.parquet")
usdinr = pl.read_parquet("../data/processed/usdinr_cleaned.parquet")

In [3]:
nifty = fe.generate_features(nifty,1.5,0.45)
gold = fe.generate_features(gold,1.75,0.5)
usdinr = fe.generate_features(usdinr,1.5,0.3)

In [4]:
print(nifty["Date"].to_list() == gold["Date"].to_list() == usdinr["Date"].to_list())

True


In [5]:
nifty.head()

Close,High,Low,Open,Date,Ret_1d,Ret_3d,Ret_5d,Ret_20d,Vol_5d,Vol_20d,FD_Close,Label,FD_Close_Lag1,MACD_Hist,RSI,ROC_10,Trend_Strength,Gap,BB_Pct,ATR_Pct,Vol_Ratio,MA_Ratio,Price_vs_MA20,Close_Pos_Range,Intraday_Return,Range_Expansion
f64,f64,f64,f64,datetime[ns],f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
4765.299805,4773.100098,4675.799805,4675.799805,2012-01-03 00:00:00,null,null,null,null,null,null,NaN,null,null,0.0,null,null,null,null,0.5,0.020419,1.0,null,null,0.919833,0.019141,1.0
4749.649902,4782.850098,4728.850098,4774.950195,2012-01-04 00:00:00,-0.003284,null,null,null,null,null,NaN,null,NaN,-0.99874,0.0,null,null,0.002025,0.5,0.01927,1.0,null,null,0.385182,-0.005299,0.554983
4749.950195,4779.799805,4730.149902,4749.0,2012-01-05 00:00:00,0.000063,null,null,null,null,null,NaN,null,NaN,-1.550935,2.024582,null,null,-0.000137,0.5,0.018094,1.0,null,null,0.398798,0.0002,0.919443
4754.100098,4794.899902,4686.850098,4724.149902,2012-01-06 00:00:00,0.000874,-0.00235,null,null,null,null,NaN,null,NaN,-1.54509,24.71013,null,null,-0.005432,0.5,0.018698,1.0,null,null,0.622398,0.00634,2.176234
4742.799805,4758.700195,4695.450195,4747.549805,2012-01-09 00:00:00,-0.002377,-0.001442,null,null,null,null,NaN,null,NaN,-2.172255,14.717186,null,null,-0.001378,0.5,0.018021,1.0,null,null,0.74861,-0.001001,0.585378


In [6]:
non_buffer_dates = [pl.datetime(2013,12,31),pl.datetime(2026,1,1)]
nifty = nifty.filter(
    (pl.col("Date")>non_buffer_dates[0]) & (pl.col("Date")<non_buffer_dates[1])
    )
gold = gold.filter(
    (pl.col("Date")>non_buffer_dates[0]) & (pl.col("Date")<non_buffer_dates[1])
    )
usdinr = usdinr.filter(
    (pl.col("Date")>non_buffer_dates[0]) & (pl.col("Date")<non_buffer_dates[1])
    )

In [7]:
nifty.head()

Close,High,Low,Open,Date,Ret_1d,Ret_3d,Ret_5d,Ret_20d,Vol_5d,Vol_20d,FD_Close,Label,FD_Close_Lag1,MACD_Hist,RSI,ROC_10,Trend_Strength,Gap,BB_Pct,ATR_Pct,Vol_Ratio,MA_Ratio,Price_vs_MA20,Close_Pos_Range,Intraday_Return,Range_Expansion
f64,f64,f64,f64,datetime[ns],f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
6221.149902,6358.299805,6211.299805,6301.25,2014-01-02 00:00:00,-0.013142,-0.014674,-0.007538,0.003112,0.007288,0.009062,297.158024,0,387.207153,-1.699671,50.006675,1.337342,0.003112,-0.000436,0.391596,0.011919,0.804234,1.005201,0.995498,0.067007,-0.012712,4.9
6211.149902,6221.700195,6171.25,6194.549805,2014-01-03 00:00:00,-0.001607,-0.012708,-0.01079,0.008148,0.007076,0.008933,321.237835,0,297.158024,-6.159605,49.154667,-0.096507,0.008148,-0.004276,0.336645,0.01143,0.792143,1.00263,0.993499,0.790877,0.00268,0.343199
6191.450195,6224.700195,6170.25,6220.850098,2014-01-06 00:00:00,-0.003172,-0.017854,-0.019378,-0.007955,0.005628,0.008455,313.973226,0,321.237835,-10.187793,47.439902,0.402168,-0.007955,0.001562,0.272652,0.01111,0.665699,0.999113,0.990741,0.38935,-0.004726,1.079286
6162.25,6221.5,6144.75,6203.899902,2014-01-07 00:00:00,-0.004716,-0.009468,-0.020481,-0.015599,0.005636,0.008469,298.305568,0,313.973226,-14.317286,44.937489,-1.785074,-0.015599,0.002011,0.190848,0.011335,0.665474,0.995767,0.986839,0.228013,-0.006714,1.409545
6174.600098,6192.100098,6160.350098,6178.049805,2014-01-08 00:00:00,0.002004,-0.005885,-0.020527,-0.029746,0.005623,0.007462,328.670374,1,298.305568,-15.609525,46.229391,-1.748745,-0.029746,0.002564,0.25607,0.01049,0.753528,0.993128,0.990318,0.448819,-0.000558,0.413681


In [8]:
nifty.columns

['Close',
 'High',
 'Low',
 'Open',
 'Date',
 'Ret_1d',
 'Ret_3d',
 'Ret_5d',
 'Ret_20d',
 'Vol_5d',
 'Vol_20d',
 'FD_Close',
 'Label',
 'FD_Close_Lag1',
 'MACD_Hist',
 'RSI',
 'ROC_10',
 'Trend_Strength',
 'Gap',
 'BB_Pct',
 'ATR_Pct',
 'Vol_Ratio',
 'MA_Ratio',
 'Price_vs_MA20',
 'Close_Pos_Range',
 'Intraday_Return',
 'Range_Expansion']

In [9]:
print(nifty["Date"].to_list() == gold["Date"].to_list() == usdinr["Date"].to_list())

True


In [10]:
nifty = nifty.with_columns(
    # Relative strength
    (pl.Series(gold["Ret_5d"]) - pl.col("Ret_5d")).alias("Gold_Nifty_RS_5d"),
    (pl.Series(gold["Ret_20d"]) - pl.col("Ret_20d")).alias("Gold_Nifty_RS_20d"),

    # USDINR features
    pl.Series(usdinr["Ret_3d"]).alias("Usdinr_Ret_3d"),
    pl.Series(usdinr["Ret_5d"]).alias("Usdinr_Ret_5d"),

    pl.when(pl.Series(usdinr["Vol_20d"]) != 0)
    .then(pl.Series(usdinr["Vol_5d"]) / pl.Series(usdinr["Vol_20d"]))
    .otherwise(1)
    .alias("Usdinr_Vol_Ratio"),

    # Momentum alignment
    (
        (pl.col("Ret_5d").sign() == pl.Series(gold["Ret_5d"]).sign())
        .cast(pl.Int8)
    ).alias("Momentum_Align"),

)
nifty = nifty.with_columns(
    # Risk-off flag
    (pl.col("Gold_Nifty_RS_5d") > 0)
    .cast(pl.Int8)
    .alias("Risk_Off"),
)

In [11]:
gold = gold.with_columns(
    # Relative strength
    (pl.Series(nifty["Ret_5d"]) - pl.col("Ret_5d")).alias("Nifty_Gold_RS_5d"),

    # USDINR features
    pl.Series(usdinr["Ret_3d"]).alias("Usdinr_Ret_3d"),
    pl.Series(usdinr["Ret_5d"]).alias("Usdinr_Ret_5d"),

    pl.when(pl.Series(usdinr["Vol_20d"]) != 0)
    .then(pl.Series(usdinr["Vol_5d"]) / pl.Series(usdinr["Vol_20d"]))
    .otherwise(1)
    .alias("Usdinr_Vol_Ratio"),

    # Momentum alignment
    (
        (pl.Series(nifty["Ret_5d"]).sign() == pl.col("Ret_5d").sign())
        .cast(pl.Int8)
    ).alias("Momentum_Align"),
)
gold = gold.with_columns(
    # Equity stress flag
    (pl.Series(nifty["Ret_5d"]) < 0)
    .cast(pl.Int8)
    .alias("Equity_Stress"),
)

In [12]:
usdinr = usdinr.with_columns(
    # External returns
    pl.Series(nifty["Ret_5d"]).alias("Nifty_Ret_5d"),
    pl.Series(gold["Ret_5d"]).alias("Gold_Ret_5d"),

    # Nifty volatility ratio
    pl.when(pl.Series(nifty["Vol_20d"]) != 0)
    .then(pl.Series(nifty["Vol_5d"]) / pl.Series(nifty["Vol_20d"]))
    .otherwise(1)
    .alias("Nifty_Vol_Ratio"),
)

usdinr = usdinr.with_columns(
    # Risk-off: gold outperforming equities
    (pl.Series(gold["Ret_5d"]) > pl.Series(nifty["Ret_5d"]))
    .cast(pl.Int8)
    .alias("Risk_Off"),

    # Equity stress: negative equity returns
    (pl.Series(nifty["Ret_5d"]) < 0)
    .cast(pl.Int8)
    .alias("Equity_Stress"),
)

In [13]:
nifty.head()

Close,High,Low,Open,Date,Ret_1d,Ret_3d,Ret_5d,Ret_20d,Vol_5d,Vol_20d,FD_Close,Label,FD_Close_Lag1,MACD_Hist,RSI,ROC_10,Trend_Strength,Gap,BB_Pct,ATR_Pct,Vol_Ratio,MA_Ratio,Price_vs_MA20,Close_Pos_Range,Intraday_Return,Range_Expansion,Gold_Nifty_RS_5d,Gold_Nifty_RS_20d,Usdinr_Ret_3d,Usdinr_Ret_5d,Usdinr_Vol_Ratio,Momentum_Align,Risk_Off
f64,f64,f64,f64,datetime[ns],f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8
6221.149902,6358.299805,6211.299805,6301.25,2014-01-02 00:00:00,-0.013142,-0.014674,-0.007538,0.003112,0.007288,0.009062,297.158024,0,387.207153,-1.699671,50.006675,1.337342,0.003112,-0.000436,0.391596,0.011919,0.804234,1.005201,0.995498,0.067007,-0.012712,4.9,0.005485,-0.040177,-0.000808,0.0,0.408289,1,1
6211.149902,6221.700195,6171.25,6194.549805,2014-01-03 00:00:00,-0.001607,-0.012708,-0.01079,0.008148,0.007076,0.008933,321.237835,0,297.158024,-6.159605,49.154667,-0.096507,0.008148,-0.004276,0.336645,0.01143,0.792143,1.00263,0.993499,0.790877,0.00268,0.343199,0.012166,-0.041547,0.006324,0.00437,0.478696,0,1
6191.450195,6224.700195,6170.25,6220.850098,2014-01-06 00:00:00,-0.003172,-0.017854,-0.019378,-0.007955,0.005628,0.008455,313.973226,0,321.237835,-10.187793,47.439902,0.402168,-0.007955,0.001562,0.272652,0.01111,0.665699,0.999113,0.990741,0.38935,-0.004726,1.079286,0.026432,-0.028491,0.007284,0.005494,0.554891,0,1
6162.25,6221.5,6144.75,6203.899902,2014-01-07 00:00:00,-0.004716,-0.009468,-0.020481,-0.015599,0.005636,0.008469,298.305568,0,313.973226,-14.317286,44.937489,-1.785074,-0.015599,0.002011,0.190848,0.011335,0.665474,0.995767,0.986839,0.228013,-0.006714,1.409545,0.033896,-0.014197,0.0076,0.010378,0.214608,0,1
6174.600098,6192.100098,6160.350098,6178.049805,2014-01-08 00:00:00,0.002004,-0.005885,-0.020527,-0.029746,0.005623,0.007462,328.670374,1,298.305568,-15.609525,46.229391,-1.748745,-0.029746,0.002564,0.25607,0.01049,0.753528,0.993128,0.990318,0.448819,-0.000558,0.413681,0.028825,0.00293,0.000322,0.004856,0.590477,0,1


In [14]:
# Assuming you named your pandas Series 'frac_diff_close'
print(f"Number of NaN values in Nifty: {nifty.null_count().to_numpy().sum()}")
print(f"Number of NaN values in Gold: {gold.null_count().to_numpy().sum()}")
print(f"Number of NaN values in UsdInr: {usdinr.null_count().to_numpy().sum()}")


Number of NaN values in Nifty: 0
Number of NaN values in Gold: 0
Number of NaN values in UsdInr: 0


In [15]:
nifty["Label"].value_counts()

Label,count
i64,u32
0,1018
1,886
-1,1035


In [16]:
temporal_split = [
    [pl.datetime(2013,12,31),pl.datetime(2024,1,1)],
    [pl.datetime(2024,2,29),pl.datetime(2026,1,1)]
    ]
nifty_train = nifty.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
nifty_test = nifty.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )

gold_train = gold.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
gold_test = gold.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )
usdinr_train = usdinr.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
usdinr_test = usdinr.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )

In [17]:
nifty_train.write_parquet("../data/processed/train/nifty.parquet")
gold_train.write_parquet("../data/processed/train/gold.parquet")
usdinr_train.write_parquet("../data/processed/train/usdinr.parquet")
nifty_test.write_parquet("../data/processed/test/nifty.parquet")
gold_test.write_parquet("../data/processed/test/gold.parquet")
usdinr_test.write_parquet("../data/processed/test/usdinr.parquet")


In [18]:
nifty.shape

(2939, 34)